In [1]:
import re
import pandas as pd
import numpy as np
from deep_translator import GoogleTranslator

from config import RESSERENO_ROOT

## Experts and high school 2019 data

In [3]:
def cue_word_extraction(file):
    data = open(file).readlines()

    pairs = [re.findall(r'"([^"]+)"', line) for line in data]

    df = pd.DataFrame(pairs, columns=['association_0', 'association_1'])

    counts_0 = df['association_0'].value_counts()
    counts_1 = df['association_1'].value_counts()

    cue_words = counts_0[:len(counts_0) // 2]

    cue_words_list = cue_words.index.tolist()

    return df, cue_words_list, counts_0, counts_1

In [ ]:
experts_df_unfiltered, cue_words_experts, counts_0, counts_1 = cue_word_extraction(f'{RESSERENO_ROOT}/data/data_2019/Experts_unfiltered.txt')

In [5]:
combined = counts_0.add(counts_1, fill_value=0)
list1 = combined.sort_values(ascending=False)[:50].keys()
list1

Index(['science', 'physics', 'university', 'system', 'school', 'mathematics',
       'life', 'network', 'complex', 'chemistry', 'art', 'model', 'biology',
       'theory', 'conference', 'water', 'statistics', 'brain', 'electricity',
       'scientific', 'analysis', 'computer', 'studies', 'word', 'market',
       'evolution', 'language', 'loop', 'teaching', 'number', 'integral',
       'chaos', 'degree', 'solid', 'dna', 'cloud', 'experiment', 'cell',
       'derivative', 'space', 'graph', 'person', 'lab', 'mathematician',
       'prisoner', 'game', 'theorem', 'student', 'trigonometry', 'hub'],
      dtype='object')

In [ ]:
students_df_unfiltered, cue_words_students_unfiltered, count3, count4 = cue_word_extraction(f'{ROOT}/data/data_2019/Students_unFiltered.txt')

In [12]:
combined = count3.add(count4, fill_value=0)
list2 = combined.sort_values(ascending=False)[:50].keys()

list_en = [
    GoogleTranslator(source='it', target='en').translate(word) for word in list2
]

In [13]:
# list2 = list(counts_0[:50].keys())
len(set(list1).intersection(set(list_en)))
set(list1).intersection(set(list_en))

{'art',
 'brain',
 'cell',
 'chemistry',
 'derivative',
 'evolution',
 'experiment',
 'graph',
 'life',
 'mathematics',
 'physics',
 'school',
 'science',
 'system',
 'theory'}

In [ ]:
students_df_filtered, cue_words_students_filtered,_, _ = cue_word_extraction(f'{RESSERENO_ROOT}/data/data_2019/Students_Filtered.txt')

### Check if valences are for filtered or unfiltered df

In [ ]:
# counts how many valence ratings per word exist in ValenceStudentsUPD.txt
counts = {}
with open(f"{RESSERENO_ROOT}/data/data_2019/ValenceStudentsUPD.txt", "r") as f:
    for line in f:
        parts = line.strip().split()
        # gets the first element of each row, which is the word
        word = parts[0]
        # gets all the numbers after the word, removing the .
        numbers = [p for p in parts[1:] if p.replace('.', '').isdigit()]
        # counts how many valence ratings per word exist
        counts[word] = counts.get(word, 0) + len(numbers)
df_counts = pd.DataFrame(list(counts.items()), columns=['word', 'count_valence'], index=None)

In [ ]:
def count_students_valence(df, file_name):
    combined_counts = (
        df['association_0'].value_counts()
        .add(df['association_1'].value_counts(), fill_value=0)
        .astype(int)
    )

    combined_counts = pd.DataFrame(list(combined_counts.items()), columns=['word', 'count_cue_ass_word'], index=None)

    df = pd.merge(df_counts, combined_counts, on='word', how='outer')
    df.to_csv(f'{RESSERENO_ROOT}/out/formatting/students_2019/combined_counts_{file_name}.csv', index=False)

In [18]:
count_students_valence(students_df_unfiltered, 'unfiltered')
count_students_valence(students_df_filtered, 'filtered')

### Creation df for word and associations

In [19]:
# Assign valences sequentially
def assign_valence(row, col_name, valence_map):
    word = row[col_name]
    if word in valence_map and len(valence_map[word]) > 0:
        return float(valence_map[word].pop(0).replace('.', ''))
    else:
        return np.nan

In [ ]:
def association_valence_df(input, df, output):
    valences = open(f"{RESSERENO_ROOT}/data/data_2019/{input}.txt", "r").readlines()
    valences = [v.strip().split('\t') for v in valences]

    valence_map = {}
    for row in valences:
        if not row[0].replace('.', '').isdigit():
            word = row[0]
            vals = [v for v in row[1:] if v.replace('.', '').isdigit()]
            valence_map.setdefault(word, []).extend(vals)

    df['valence_association_1'] = df.apply(assign_valence, axis=1, args=(('association_1', valence_map)))
    df['valence_association_0'] = df.apply(assign_valence, axis=1, args=(('association_0', valence_map)))

    # df = df['association_0', 'valence_association_0', 'association_1', 'valence_association_1']

    df['new_index'] = df.index + 1
    df['cue_word_original'] = df['association_0']

    # melt from wide to long
    df_long = pd.wide_to_long(df, stubnames=['association', 'valence_association'], i=['new_index', 'cue_word_original'], j='index', sep='_', suffix=r'\d+').reset_index()

    # optional: rename columns for clarity
    df_long = df_long.rename(columns={
        'association': 'word',
        'valence_association': 'valence', 
        'cue_word_original': 'cue_word'
    })

    columns_to_clean = ['cue_word', 'word']
    for col in columns_to_clean:
        if col in df_long.columns:
            df_long[col] = (
                df_long[col]
                .astype(str)
                .str.lower()
                .str.replace(r'[\d\W_]+', ' ', regex=True)
                .str.strip()
            )
    df_long = df_long.dropna()

    df_long = df_long[(df_long['word'] != 'nan') & (df_long['word'] != '')]
    df_long.to_csv(f'{RESSERENO_ROOT}/out/formatting/{output}/associations_valence_{output}.csv', index=False)

    return df_long

In [21]:
students_ass_val_df = association_valence_df('ValenceStudentsUPD', students_df_unfiltered, 'students_2019')

In [22]:
experts_ass_val_df = association_valence_df('ValenceExperts', experts_df_unfiltered, 'experts_2019')